In [1]:
from importlib.metadata import version
import tiktoken

print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.12.0


In [4]:
tokenizer = tiktoken.get_encoding("gpt2")
text = ("Hello, world! <|endoftext|> This is a test."
        " Let's see how the tokenizer handles special tokens.")

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print("Encoded integers:", integers)

Encoded integers: [15496, 11, 995, 0, 220, 50256, 770, 318, 257, 1332, 13, 3914, 338, 766, 703, 262, 11241, 7509, 17105, 2041, 16326, 13]


In [7]:
strings = tokenizer.decode(integers)
print("Decoded string:", strings)

Decoded string: Hello, world! <|endoftext|> This is a test. Let's see how the tokenizer handles special tokens.


In [ ]:
# 未知のトークンを含む場合の動作確認(Akwirw ier)
tokenizer = tiktoken.get_encoding("gpt2")
text = ("Hello, world! <|endoftext|> This is a test. Akwirw ier."
        " Let's see how the tokenizer handles special tokens.")

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print("Encoded integers:", integers)

strings = tokenizer.decode(integers)
print("Decoded string:", strings)

Encoded integers: [15496, 11, 995, 0, 220, 50256, 770, 318, 257, 1332, 13, 9084, 86, 343, 86, 220, 959, 13, 3914, 338, 766, 703, 262, 11241, 7509, 17105, 2041, 16326, 13]
Decoded string: Hello, world! <|endoftext|> This is a test. Akwirw ier. Let's see how the tokenizer handles special tokens.


In [10]:
with open("verdict.txt", "r") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print("Number of tokens in verdict.txt:", len(enc_text))

Number of tokens in verdict.txt: 5145


In [13]:
enc_sample = enc_text[50:]

context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]
print("x:", x)
print("y:", y)

for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(f"Context: {context}, String: '{tokenizer.decode(context)}' -> Next token: {desired}, String: '{tokenizer.decode([desired])}'")

x: [290, 4920, 2241, 287]
y: [4920, 2241, 287, 257]
Context: [290], String: ' and' -> Next token: 4920, String: ' established'
Context: [290, 4920], String: ' and established' -> Next token: 2241, String: ' himself'
Context: [290, 4920, 2241], String: ' and established himself' -> Next token: 287, String: ' in'
Context: [290, 4920, 2241, 287], String: ' and established himself in' -> Next token: 257, String: ' a'


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(text)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk, dtype=torch.long))
            self.target_ids.append(torch.tensor(target_chunk, dtype=torch.long))
    
    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader(text, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(text, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataloader 